In [ ]:
!pip install timm dropbox albumentations pandas opencv-python scikit-learn

import os
import cv2
import re
import time
import random
import numpy as np
import pandas as pd
import dropbox
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, confusion_matrix
)
from tqdm.notebook import tqdm

class Config:
    SIIM_CSV = '../input/siim-isic-melanoma-classification/train.csv'
    SIIM_DIR = '../input/siim-isic-melanoma-classification/jpeg/train/'
    HASNAIN_DIR = '../input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train/'
    
    MODEL_NAME = 'coatnet_0_rw_224' 
    IMG_SIZE = 224
    BATCH_SIZE = 64          
    EPOCHS = 20              
    LR = 3e-4                
    SEED = 42
    NUM_WORKERS = 4
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DROPBOX_CLUSTER_CREDS = []

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)

class DropboxCluster:
    def __init__(self, creds_list):
        self.clients = []
        print(f"🔄 Initializing Dropbox Cluster...")
        for cred in creds_list:
            try:
                dbx = dropbox.Dropbox(
                    app_key=cred['app_key'],
                    app_secret=cred['app_secret'],
                    oauth2_refresh_token=cred['refresh_token']
                )
                dbx.users_get_current_account()
                self.clients.append({'name': cred['name'], 'dbx': dbx})
            except: pass

    def upload_file(self, local_path, remote_name):
        for node in self.clients:
            try:
                with open(local_path, "rb") as f:
                    node['dbx'].files_upload(f.read(), f"/{remote_name}", mode=dropbox.files.WriteMode('overwrite'))
                print(f"   ☁️ [Saved] {remote_name} to Dropbox")
                break 
            except Exception as e:
                print(f"   ⚠️ Upload failed: {e}")

    def get_latest_checkpoint(self):
        best_checkpoint = None
        max_epoch = -1
        best_auc_found = 0.0

        print("   🔍 Checking Dropbox for existing checkpoints...")
        for node in self.clients:
            try:
                files = node['dbx'].files_list_folder('').entries
                for entry in files:
                    match = re.search(r'fused_model_epoch_(\d+)_auc_([\d\.]+).pth', entry.name)
                    if match:
                        epoch = int(match.group(1))
                        auc = float(match.group(2))
                        
                        if epoch > max_epoch:
                            max_epoch = epoch
                            best_auc_found = auc
                            best_checkpoint = {
                                'file': entry.name,
                                'epoch': epoch,
                                'auc': auc,
                                'dbx': node['dbx'],
                                'path_lower': entry.path_lower
                            }
            except: continue
        
        return best_checkpoint

def prepare_fused_dataset():
    print("🔄 Initializing Data Fusion Engine...")
    
    print("   1️⃣ Processing SIIM-ISIC Dataset...")
    if os.path.exists(Config.SIIM_CSV):
        df_siim = pd.read_csv(Config.SIIM_CSV)
        df_siim['path'] = df_siim['image_name'].apply(lambda x: os.path.join(Config.SIIM_DIR, f"{x}.jpg"))
        df_siim = df_siim[['path', 'target']]
        df_siim['source'] = 'SIIM'
        print(f"      ✅ Loaded {len(df_siim)} images from SIIM.")
    else:
        print("      ❌ SIIM CSV not found. Skipping.")
        df_siim = pd.DataFrame(columns=['path', 'target', 'source'])

    print("   2️⃣ Processing Hasnain Dataset...")
    hasnain_samples = []
    if os.path.exists(Config.HASNAIN_DIR):
        for label_name in ['benign', 'malignant']:
            folder_path = os.path.join(Config.HASNAIN_DIR, label_name)
            if os.path.exists(folder_path):
                target = 1 if label_name == 'malignant' else 0
                for img_name in os.listdir(folder_path):
                    hasnain_samples.append({
                        'path': os.path.join(folder_path, img_name),
                        'target': target,
                        'source': 'HASNAIN'
                    })
    
    df_hasnain = pd.DataFrame(hasnain_samples)
    if len(df_hasnain) > 0:
        print(f"      ✅ Loaded {len(df_hasnain)} images from Hasnain.")
    else:
        print("      ❌ Hasnain dataset folder not found. Skipping.")

    full_df = pd.concat([df_siim, df_hasnain]).reset_index(drop=True)
    n_pos = full_df['target'].sum()
    print(f"\n   🔥 DATA FUSION COMPLETE: {len(full_df)} Images (Malignant: {int(n_pos)})")
    return full_df

def get_transforms(phase):
    if phase == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=180, p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.75),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
            A.CoarseDropout(max_holes=8, max_height=20, max_width=20, min_holes=1, min_height=10, min_width=10, fill_value=0, p=0.5),
            A.OneOf([A.GaussianBlur(blur_limit=3, p=0.3), A.ISONoise(p=0.3)], p=0.5),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])

class UnifiedMelanomaDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        try:
            img = cv2.imread(path)
            if img is None: raise ValueError("Read Failed")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except:
            img = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        return img, torch.tensor(row['target'], dtype=torch.float32)

class MelanoCoAtNet(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True, num_classes=1)
    def forward(self, x): return self.model(x)

def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    running_loss = 0.0
    pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch}")
    for step, (images, labels) in pbar:
        images = images.to(Config.DEVICE)
        labels = labels.to(Config.DEVICE)
        with torch.amp.autocast('cuda'):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        running_loss += loss.item()
        pbar.set_postfix(loss=running_loss / (step + 1))
    return running_loss / len(loader)

def validate(model, loader):
    model.eval()
    preds, val_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(Config.DEVICE)
            with torch.amp.autocast('cuda'):
                outputs = model(images).squeeze(1)
            preds.append(torch.sigmoid(outputs).float().cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    
    preds = np.concatenate(preds)
    val_labels = np.concatenate(val_labels)
    try:
        auc = roc_auc_score(val_labels, preds)
        pred_labels = (preds > 0.5).astype(int)
        acc = accuracy_score(val_labels, pred_labels)
        rec = recall_score(val_labels, pred_labels, zero_division=0)
        f1 = f1_score(val_labels, pred_labels, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(val_labels, pred_labels).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        return {"auc": auc, "acc": acc, "rec": rec, "f1": f1, "spec": spec}
    except:
        return {"auc": 0.5, "acc": 0, "rec": 0, "f1": 0, "spec": 0}

def main():
    dbx = DropboxCluster(DROPBOX_CLUSTER_CREDS)
    
    df = prepare_fused_dataset()
    if len(df) == 0: return

    train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['target'], random_state=Config.SEED)
    
    count_class_0 = train_df[train_df['target'] == 0].shape[0]
    count_class_1 = train_df[train_df['target'] == 1].shape[0]
    weight_0 = 1.0 / count_class_0
    weight_1 = 1.0 / count_class_1
    sample_weights = [weight_1 if t == 1 else weight_0 for t in train_df['target']]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True)
    
    train_loader = DataLoader(UnifiedMelanomaDataset(train_df, get_transforms('train')),
                              batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=Config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(UnifiedMelanomaDataset(val_df, get_transforms('val')),
                            batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    print(f"🚀 Initializing {Config.MODEL_NAME}...")
    model = MelanoCoAtNet(Config.MODEL_NAME).to(Config.DEVICE)
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    
    optimizer = optim.AdamW(model.parameters(), lr=Config.LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda')
    
    start_epoch = 1
    best_auc = 0.0
    
    latest_ckpt = dbx.get_latest_checkpoint()
    if latest_ckpt:
        print(f"\n   📥 FOUND CHECKPOINT: {latest_ckpt['file']}")
        print(f"      Downloading to resume training...")
        try:
            latest_ckpt['dbx'].files_download_to_file(latest_ckpt['file'], latest_ckpt['path_lower'])
            
            checkpoint = torch.load(latest_ckpt['file'], map_location=Config.DEVICE)
            
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
                
            start_epoch = latest_ckpt['epoch'] + 1
            best_auc = latest_ckpt['auc']
            
            print(f"   ✅ RESUMING successfully from Epoch {start_epoch}")
            print(f"      Previous Best AUC: {best_auc:.4f}")
            
            for _ in range(start_epoch - 1): scheduler.step()
            
        except Exception as e:
            print(f"      ❌ Resume Failed: {e}. Starting from Scratch.")
    else:
        print("\n   🆕 No previous checkpoints found. Starting from Epoch 1.")

    if start_epoch > Config.EPOCHS:
        print("   🎉 Training already complete! (Start Epoch > Max Epochs)")
        return

    print("🔥 Starting Training...")
    for epoch in range(start_epoch, Config.EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
        metrics = validate(model, val_loader)
        
        print(f"\n📊 EPOCH {epoch} REPORT:")
        print(f"   -------------------------")
        print(f"   📉 Loss        : {loss:.4f}")
        print(f"   🏆 AUC         : {metrics['auc']:.4f}")
        print(f"   🎯 Accuracy    : {metrics['acc']:.4f}")
        print(f"   🩺 Sensitivity : {metrics['rec']:.4f}")
        print(f"   🛡️ Specificity : {metrics['spec']:.4f}")
        print(f"   ⚖️ F1-Score    : {metrics['f1']:.4f}")
        print(f"   -------------------------")
        
        if metrics['auc'] > best_auc:
            best_auc = metrics['auc']
            filename = f"fused_model_epoch_{epoch}_auc_{metrics['auc']:.4f}.pth"
            torch.save(model.state_dict(), filename)
            print(f"   💾 Saved Best Model: {filename}")
            dbx.upload_file(filename, filename)
            
        scheduler.step()

if __name__ == '__main__':
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 10.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

🔄 Initializing Dropbox Cluster...
🔄 Initializing Data Fusion Engine...
   1️⃣ Processing SIIM-ISIC Dataset...
      ✅ Loaded 33126 images from SIIM.
   2️⃣ Processing Hasnain Dataset...
      ✅ Loaded 9605 images from Hasnain.

   🔥 DATA FUSION COMPLETE: 42731 Images (Malignant: 5189)
🚀 Initializing coatnet_0_rw_224...


/tmp/ipykernel_55/1944711428.py:197: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=20, max_width=20, min_holes=1, min_height=10, min_width=10, fill_value=0, p=0.5),


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

...

Epoch 12:   0%|          | 0/568 [00:00<?, ?it/s]

Validating:   0%|          | 0/101 [00:00<?, ?it/s]


📊 EPOCH 12 REPORT:
   -------------------------
   📉 Loss        : 0.1014
   🏆 AUC         : 0.9865
   🎯 Accuracy    : 0.9315
   🩺 Sensitivity : 0.9422
   🛡️ Specificity : 0.9300
   ⚖️ F1-Score    : 0.7696
   -------------------------


Epoch 13:   0%|          | 0/568 [00:00<?, ?it/s]

KeyboardInterrupt: 